# 07 · Sorting algorithms

After this notebook you can implement and compare the classic sorts (bubble, selection, insertion, merge, quick, heap, counting), explain stability and quicksort's worst case, say what Python's Timsort does, sort by several keys with `key=` (including reversing just one of them), and find the k-th largest element with quickselect.

**Time:** ~45 min · **Runs:** offline on CPU in < 1 min · **Needs:** [06 · Binary search](06_binary_search.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 10 "Sorting & searching"

## Why this matters in interviews

- "Sort first" is the opening move of many optimal solutions (two pointers, intervals, greedy, dedupe), so you must know it costs O(n log n) and what that buys.
- Classic follow-ups test understanding, not memorisation: *why is quicksort O(n²) in the worst case? what does stable mean and when do you need it? why is insertion sort used inside Timsort?*
- Practical Python matters too: multi-key sorting with `key=`, `itemgetter`, and `cmp_to_key` come up in almost every data-wrangling round — and in GenAI work (ranking retrieved chunks by score, then by recency).

## What interviewers ask

- "Implement merge sort / quicksort. What are their time and space complexities?"
- "What is quicksort's worst case, when does it happen, and how do you avoid it?"
- "What does *stable* mean? Which sorts are stable? Why does it matter for multi-key sorting?"
- "What algorithm does Python use? Why is sorting already-sorted data fast?"
- "Sort by frequency, then by value descending." (LeetCode 1636) · "Largest number from these digits." (LC 179)
- "Find the k-th largest element — can you beat O(n log n)?" (LC 215)
- "Count the inversions in an array." (merge-sort follow-up) · "Sort integers in a small range in O(n)." (counting sort)

In [ ]:
import heapq
import random
import sys
import time
from collections import Counter
from functools import cmp_to_key
from itertools import permutations
from operator import itemgetter

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kendalltau

random.seed(0)
rng = random.Random(1)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})
sys.setrecursionlimit(10_000)      # the naive quicksort demo recurses n deep on sorted input (safe in Python 3.12)


def identity(x):
    return x

## The landscape

| Algorithm | Best | Average | Worst | Extra space | Stable | Notes |
|---|---|---|---|---|---|---|
| Bubble (early exit) | O(n) | O(n²) | O(n²) | O(1) | yes | teaching only |
| Selection | O(n²) | O(n²) | O(n²) | O(1) | **no** | fewest swaps (n − 1) |
| Insertion | O(n) | O(n²) | O(n²) | O(1) | yes | fastest for tiny or nearly sorted inputs |
| Merge | O(n log n) | O(n log n) | O(n log n) | O(n) | yes | predictable; good for linked lists and external sorting |
| Quick | O(n log n) | O(n log n) | **O(n²)** | O(log n) stack | no | fastest in practice with a random pivot |
| Heap | O(n log n) | O(n log n) | O(n log n) | O(1) in place | no | guaranteed bound, poor cache behaviour |
| Counting | O(n + k) | O(n + k) | O(n + k) | O(n + k) | yes | integer keys in a small range [0, k) |
| **Timsort** (Python) | **O(n)** | O(n log n) | O(n log n) | O(n) | yes | merges natural runs; insertion sort for short runs |

Comparison sorts cannot beat **Ω(n log n)** comparisons in the worst case (there are n! orderings and each comparison gives one bit, log₂ n! ≈ n log₂ n). Counting and radix sort escape the bound because they do not compare — they index by the key.

**Recognise "sort first" when you see:** pairs or triplets with a target (then two pointers), overlapping intervals, "closest", grouping duplicates, greedy choices ("always take the smallest / earliest"), or many searches over the same data (then binary search).

## The O(n²) sorts, step by step

Three simple sorts, each able to record snapshots of the array as it goes:

In [ ]:
def bubble_sort(a, key=identity, snapshots=None):
    """Swap adjacent out-of-order pairs; each pass bubbles the largest remaining value to the end."""
    a = list(a)
    for end in range(len(a) - 1, 0, -1):
        swapped = False
        for i in range(end):
            if key(a[i]) > key(a[i + 1]):          # strict: equal items never swap -> stable
                a[i], a[i + 1] = a[i + 1], a[i]
                swapped = True
        if snapshots is not None:
            snapshots.append(a.copy())
        if not swapped:                            # early exit: O(n) on sorted input
            break
    return a


def selection_sort(a, key=identity, snapshots=None):
    """Find the minimum of the unsorted part and swap it to the front."""
    a = list(a)
    for i in range(len(a) - 1):
        m = min(range(i, len(a)), key=lambda j: key(a[j]))
        a[i], a[m] = a[m], a[i]                    # the long-distance swap breaks stability
        if snapshots is not None:
            snapshots.append(a.copy())
    return a


def insertion_sort(a, key=identity, snapshots=None):
    """Grow a sorted prefix: shift bigger items right and drop the next item into its gap."""
    a = list(a)
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and key(a[j]) > key(x):       # shifts, not swaps; stops at equals -> stable
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = x
        if snapshots is not None:
            snapshots.append(a.copy())
    return a


for sort in (bubble_sort, selection_sort, insertion_sort):
    for case in [[], [1], [2, 1], [5, 2, 8, 1, 9, 3, 7, 4], [3, 3, 1, 3]]:
        assert sort(case) == sorted(case), (sort.__name__, case)
    for _ in range(200):
        case = [rng.randint(0, 20) for _ in range(rng.randint(0, 25))]
        assert sort(case) == sorted(case)
print("bubble, selection and insertion sort agree with sorted() on edge cases + 200 random lists each")

In [ ]:
values = [5, 2, 8, 1, 9, 3, 7, 4]
fig, axes = plt.subplots(3, 5, figsize=(11, 4.5), sharey=True)
for row, (sort, done_region) in enumerate([(bubble_sort, "suffix"), (selection_sort, "prefix"), (insertion_sort, "prefix+1")]):
    snaps = []
    sort(values, snapshots=snaps)
    frames = [values] + snaps[:3] + [snaps[-1]]
    labels = ["start", "after step 1", "after step 2", "after step 3", f"done ({len(snaps)} steps)"]
    for col, (frame, label) in enumerate(zip(frames, labels)):
        k = col if col < 4 else len(values)                      # how many positions are settled
        settled = {"suffix": range(len(values) - k, len(values)), "prefix": range(k),
                   "prefix+1": range(min(k + 1, len(values)) if col else 0)}[done_region]
        colors = [COLORS[0] if i in settled else "#c3c2b7" for i in range(len(frame))]
        ax = axes[row, col]
        ax.bar(range(len(frame)), frame, color=colors, width=0.75)
        ax.set_xticks([])
        ax.grid(False)
        if row == 0:
            ax.set_title(label, fontsize=9)
        if col == 0:
            ax.set_ylabel(sort.__name__.replace("_sort", ""), fontsize=10)
fig.suptitle("O(n²) sorts: the settled region (blue) grows by one element per step")
plt.tight_layout()
plt.show()

Bubble sort settles the **end** of the array first (the largest value bubbles right each pass); selection sort settles the **front** (the minimum is swapped in); insertion sort keeps a sorted **prefix** that is not yet final — later small values still get inserted into it.

## Merge sort — divide and conquer

**In plain English:** split the list in half, sort each half (recursively), then **merge** the two sorted halves by repeatedly taking the smaller front element (the two-pointer merge from [notebook 02](02_arrays_and_two_pointers.ipynb)). There are log₂ n levels of halving and each level does O(n) merging work: O(n log n) always, with O(n) extra memory for the merged lists.

In [ ]:
def merge_sort(a, key=identity):
    if len(a) <= 1:
        return list(a)
    mid = len(a) // 2
    left, right = merge_sort(a[:mid], key), merge_sort(a[mid:], key)
    merged, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):          # <= takes from the left on ties -> stable
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    return merged + left[i:] + right[j:]


for _ in range(300):
    case = [rng.randint(-50, 50) for _ in range(rng.randint(0, 40))]
    assert merge_sort(case) == sorted(case)
print("merge_sort agrees with sorted() on 300 random lists")

### Classic problem · Count inversions with merge sort

**Statement:** count pairs `i < j` with `a[i] > a[j]` — "how far from sorted is this?". `[2, 4, 1, 3, 5]` → 3. **Brute force:** check all pairs, O(n²). **Optimal:** during the merge, whenever an element of the *right* half is taken before the remaining elements of the left half, it forms an inversion with each of them: add `len(left) - i`. O(n log n).

This is also how ranking agreement is measured: **Kendall's τ** between two rankings (say, a reranker's order vs. the retriever's order) is `1 − 2·inversions / (n(n−1)/2)`. The assert below checks that against `scipy.stats.kendalltau`.

In [ ]:
def sort_and_count(a):
    if len(a) <= 1:
        return list(a), 0
    mid = len(a) // 2
    left, inv_left = sort_and_count(a[:mid])
    right, inv_right = sort_and_count(a[mid:])
    merged, i, j, inversions = [], 0, 0, inv_left + inv_right
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            inversions += len(left) - i            # right[j] jumps ahead of every remaining left element
            j += 1
    return merged + left[i:] + right[j:], inversions


def inversions_brute(a):
    return sum(a[i] > a[j] for i in range(len(a)) for j in range(i + 1, len(a)))


assert sort_and_count([2, 4, 1, 3, 5])[1] == 3 and sort_and_count([])[1] == 0 and sort_and_count([5, 4, 3, 2, 1])[1] == 10
for _ in range(300):
    case = [rng.randint(0, 30) for _ in range(rng.randint(0, 30))]
    assert sort_and_count(case)[1] == inversions_brute(case)

retriever_order = list(range(10))                        # document ids in the retriever's order
reranked = [2, 0, 1, 3, 5, 4, 9, 6, 7, 8]                  # the reranker's order of the same documents
n_docs = len(reranked)
tau = 1 - 2 * sort_and_count(reranked)[1] / (n_docs * (n_docs - 1) / 2)
print(f"inversions = {sort_and_count(reranked)[1]}, Kendall tau = {tau:.3f}, scipy says {kendalltau(retriever_order, reranked).statistic:.3f}")
assert abs(tau - kendalltau(retriever_order, reranked).statistic) < 1e-12

## Quicksort — and its worst case

**In plain English:** pick a **pivot**, partition the array so smaller values go left and larger go right, then sort the two sides recursively. It sorts in place and is very fast in practice — *if* the pivot splits the array roughly in half. If the pivot is always the smallest or largest value, one side is empty every time: n levels of n work, **O(n²)**, and recursion n deep.

That is exactly what happens with the textbook "last element as pivot" on **already-sorted input** — common in real data. The fix: a **random pivot** (or median-of-three), which makes the bad case astronomically unlikely: expected O(n log n) on every input. Many equal keys need a three-way partition (the [Dutch flag](02_arrays_and_two_pointers.ipynb)).

In [ ]:
def quick_sort(a, pivot="random", seed=0):
    """Lomuto partition. Returns (sorted list, stats) so the worst case can be measured."""
    a, stats, pick = list(a), {"comparisons": 0, "max_depth": 0}, random.Random(seed)

    def partition(lo, hi):
        if pivot == "random":
            r = pick.randint(lo, hi)
            a[r], a[hi] = a[hi], a[r]              # move a random element into the pivot slot
        p, i = a[hi], lo
        for j in range(lo, hi):
            stats["comparisons"] += 1
            if a[j] < p:
                a[i], a[j] = a[j], a[i]
                i += 1
        a[i], a[hi] = a[hi], a[i]                  # the pivot lands in its final position
        return i

    def sort(lo, hi, depth):
        if lo >= hi:
            return
        stats["max_depth"] = max(stats["max_depth"], depth)
        p = partition(lo, hi)
        sort(lo, p - 1, depth + 1)
        sort(p + 1, hi, depth + 1)

    sort(0, len(a) - 1, 1)
    return a, stats


for _ in range(300):
    case = [rng.randint(0, 50) for _ in range(rng.randint(0, 40))]
    assert quick_sort(case, "last")[0] == quick_sort(case, "random")[0] == sorted(case)

n = 800
shuffled = random.Random(3).sample(range(n), n)
print(f"{'input':>10} {'pivot':>7} {'comparisons':>12} {'recursion depth':>16}")
for name, data in [("random", shuffled), ("sorted", sorted(shuffled))]:
    for pivot in ("last", "random"):
        stats = quick_sort(data, pivot)[1]
        print(f"{name:>10} {pivot:>7} {stats['comparisons']:>12,} {stats['max_depth']:>16}")
worst = quick_sort(range(n), "last")[1]
assert worst["comparisons"] == n * (n - 1) // 2 and worst["max_depth"] == n - 1    # exactly the O(n²) worst case
assert quick_sort(range(n), "random")[1]["comparisons"] < 5 * n * np.log2(n)
print(f"\nn log2 n = {n * np.log2(n):,.0f};  n(n-1)/2 = {n * (n - 1) // 2:,}")

Sorted input with a last-element pivot does *exactly* n(n−1)/2 comparisons and recurses n − 1 deep (in C that is a stack overflow; in Python a `RecursionError` at the default limit of 1000). A random pivot brings it back to about n log₂ n comparisons at this size (the exact expectation is 2n ln n − O(n), which tends to 1.39 · n log₂ n for large n) and a recursion depth of about 20.

## Heap sort and counting sort

**Heap sort:** build a min-heap in O(n) (`heapify`), then pop the minimum n times at O(log n) each. O(n log n) guaranteed — but not stable, and slower in practice than merge or quick sort because heap accesses jump around memory. More on heaps in [10 · Heaps and top-k](10_heaps_and_top_k.ipynb).

**Counting sort:** when keys are small integers in `[0, k)` — ages, grades, ratings — count how many of each key, turn the counts into starting positions (prefix sums), and place each item at its key's next free slot. O(n + k), no comparisons at all. Placing items left to right keeps it **stable**, which is what lets radix sort run it once per digit.

In [ ]:
def heap_sort(a):
    h = list(a)
    heapq.heapify(h)                                     # O(n)
    return [heapq.heappop(h) for _ in range(len(h))]     # n pops, O(log n) each


def counting_sort(items, key=identity, k=None):
    """Stable sort of items whose key(item) is an int in [0, k)."""
    k = k if k is not None else max(map(key, items), default=-1) + 1
    counts = [0] * k
    for x in items:
        counts[key(x)] += 1
    starts, total = [0] * k, 0
    for v in range(k):                                   # prefix sums: where each key's block starts
        starts[v], total = total, total + counts[v]
    out = [None] * len(items)
    for x in items:                                      # left to right -> equal keys keep their order
        out[starts[key(x)]] = x
        starts[key(x)] += 1
    return out


ages = [rng.randint(0, 120) for _ in range(1_000)]
assert heap_sort(ages) == counting_sort(ages) == sorted(ages)
assert heap_sort([]) == counting_sort([]) == []
people = [(rng.choice("ABCDE"), rng.randint(0, 4)) for _ in range(50)]            # (name initial, rating 0-4)
assert counting_sort(people, key=itemgetter(1), k=5) == sorted(people, key=itemgetter(1))   # stable, like sorted()
print("heap_sort and counting_sort match sorted(); counting_sort keeps equal ratings in their original order")

## Stability, with an example

**Stable** means items with equal keys keep their original relative order. It matters whenever you sort by one field *after* the data already has a meaningful order — e.g. a table already sorted by name, now sorted by grade: a stable sort keeps names alphabetical within each grade, so two stable passes give a two-key sort for free.

In [ ]:
students = [("ana", "B"), ("ben", "B"), ("cat", "A"), ("dev", "C")]     # already sorted by name
by_grade = itemgetter(1)
stable_expected = sorted(students, key=by_grade)                 # Python's sort is guaranteed stable
print("stable (sorted):       ", [f"{n}:{g}" for n, g in stable_expected])
print("selection sort:        ", [f"{n}:{g}" for n, g in selection_sort(students, key=by_grade)])
for sort in (bubble_sort, insertion_sort, merge_sort):
    assert sort(students, key=by_grade) == stable_expected, sort.__name__
assert counting_sort(students, key=lambda s: "ABC".index(s[1]), k=3) == stable_expected
assert selection_sort(students, key=by_grade) != stable_expected        # same grades, names out of order
print("bubble, insertion, merge and counting sort are stable here; selection sort is not")

Selection sort's first swap moved `ana` (B) to the back, past `ben` (B), and nothing moved her back. Quicksort and heap sort are unstable for the same reason.

## Python's sort: Timsort

`sorted()` and `list.sort()` use **Timsort** (Tim Peters, 2002): scan for **natural runs** (already ascending, or strictly descending — reversed in place), extend short runs to a minimum length (32–64) with binary insertion sort, then merge runs pairwise (with "galloping" when one run keeps winning). Since Python 3.11 the choice of *which* runs to merge follows the "powersort" rule. Consequences you can quote:

- **Stable**, O(n log n) worst case, O(n) extra memory.
- **O(n) on already-sorted or reverse-sorted input** — one run, no merging — and fast on "sorted plus a few appended items".
- `list.sort()` sorts in place and returns `None`; `sorted()` returns a new list and accepts any iterable.

Count the comparisons it actually makes (a wrapper class counts every `<`):

In [ ]:
class Counted:
    comparisons = 0
    __slots__ = ("value",)

    def __init__(self, value):
        self.value = value

    def __lt__(self, other):                 # sorted() only ever uses <
        Counted.comparisons += 1
        return self.value < other.value


def comparisons_to_sort(values):
    Counted.comparisons = 0
    sorted(Counted(v) for v in values)
    return Counted.comparisons


n = 10_000
base = random.Random(5).sample(range(n), n)
nearly = sorted(base)
nearly[100], nearly[5_000] = nearly[5_000], nearly[100]           # sorted, except one swapped pair
results = {"already sorted": comparisons_to_sort(sorted(base)), "reverse sorted": comparisons_to_sort(sorted(base, reverse=True)),
           "sorted + 1 swap": comparisons_to_sort(nearly), "random": comparisons_to_sort(base)}
for name, c in results.items():
    print(f"{name:>15}: {c:>7,} comparisons  ({c / n:.1f} per element)")
print(f"{'n log2 n':>15}: {n * np.log2(n):>9,.0f}")
assert results["already sorted"] == results["reverse sorted"] == n - 1         # one run: linear
assert results["random"] > 10 * n and results["sorted + 1 swap"] < 3 * n

## `key=` and multi-key sorting

Sort by *what matters* with `key=`: the key function runs once per element (decorate–sort–undecorate), then tuples compare field by field — which is what makes multi-key sorting a one-liner.

| Goal | Code |
|---|---|
| by one field | `sorted(rows, key=itemgetter("dept"))` or `key=lambda r: r["dept"]` |
| by several fields | `key=itemgetter("dept", "name")` or `key=lambda r: (r["dept"], r["name"])` |
| one **numeric** field descending | negate it: `key=lambda r: (r["dept"], -r["salary"])` |
| one **non-numeric** field descending | two stable passes: secondary key first, then primary with `reverse=True` |
| custom pairwise rule | `key=functools.cmp_to_key(compare)` |
| objects | `operator.attrgetter("score")` |

In [ ]:
staff = [{"name": "ana", "dept": "eng", "salary": 120}, {"name": "bo", "dept": "ops", "salary": 90},
         {"name": "cy", "dept": "eng", "salary": 150}, {"name": "di", "dept": "ops", "salary": 90},
         {"name": "ed", "dept": "eng", "salary": 120}]
names = lambda rows: [r["name"] for r in rows]

by_dept_name = sorted(staff, key=itemgetter("dept", "name"))
by_dept_salary_desc = sorted(staff, key=lambda r: (r["dept"], -r["salary"], r["name"]))
dept_desc_name_asc = sorted(sorted(staff, key=itemgetter("name")), key=itemgetter("dept"), reverse=True)
print("dept, then name:               ", names(by_dept_name))
print("dept, then salary high->low:   ", names(by_dept_salary_desc))
print("dept DESCENDING, then name asc:", names(dept_desc_name_asc))
assert names(by_dept_name) == ["ana", "cy", "ed", "bo", "di"]
assert names(by_dept_salary_desc) == ["cy", "ana", "ed", "bo", "di"]
assert names(dept_desc_name_asc) == ["bo", "di", "ana", "cy", "ed"]      # reverse=True keeps ties in order: still stable

### Classic problem · Sort by increasing frequency, ties by value decreasing (LC 1636)

`[1,1,2,2,2,3]` → `[3,1,1,2,2,2]`; `[2,3,1,3,2]` → `[1,3,3,2,2]`. A tuple key with one numeric field negated does it in one line: O(n log n).

In [ ]:
def frequency_sort(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))


for nums, expected in [([1, 1, 2, 2, 2, 3], [3, 1, 1, 2, 2, 2]), ([2, 3, 1, 3, 2], [1, 3, 3, 2, 2]),
                       ([-1, 1, -6, 4, 5, -6, 1, 4, 1], [5, -1, 4, 4, -6, -6, 1, 1, 1]), ([], [])]:
    assert frequency_sort(nums) == expected, nums
print("frequency_sort:", frequency_sort([2, 3, 1, 3, 2]))

### Classic problem · Largest number (LC 179)

Arrange non-negative integers to form the largest number: `[3, 30, 34, 5, 9]` → `"9534330"`. No per-element key works directly ("3" vs "30"?), but a **pairwise rule** does: put `a` before `b` if `a + b > b + a` as strings. `cmp_to_key` turns that comparison into a key. **Brute force:** try all n! orders.

In [ ]:
def largest_number(nums):
    def compare(a, b):
        return -1 if a + b > b + a else (1 if a + b < b + a else 0)
    digits = sorted(map(str, nums), key=cmp_to_key(compare))
    joined = "".join(digits)
    return "0" if joined[0] == "0" else joined              # [0, 0] -> "0", not "00"


def largest_number_brute(nums):
    return str(max(int("".join(p)) for p in permutations(map(str, nums))))


for nums, expected in [([10, 2], "210"), ([3, 30, 34, 5, 9], "9534330"), ([0, 0], "0"), ([1], "1"), ([432, 43243], "43243432")]:
    assert largest_number(nums) == expected, nums
for _ in range(200):
    nums = [rng.randint(0, 120) for _ in range(rng.randint(1, 5))]
    assert largest_number(nums) == largest_number_brute(nums)
print("largest_number agrees with trying every permutation on 200 random inputs")

## Quickselect: the k-th largest without sorting everything (LC 215)

**Statement:** the k-th largest element of an unsorted array. `[3,2,1,5,6,4]`, k=2 → 5.

| Approach | Time | Extra space |
|---|---|---|
| `sorted(nums)[-k]` | O(n log n) | O(n) |
| `heapq.nlargest(k, nums)[-1]` — a min-heap of size k | O(n log k) | O(k) |
| **quickselect** — partition, recurse into *one* side only | **O(n) average**, O(n²) worst | O(1) (iterative) |

Quickselect is quicksort that throws half away: after partitioning, the pivot is at its final sorted index `p`. If `p` is the index we want, done; otherwise continue only on the side that contains it. Expected work n + n/2 + n/4 + … ≈ 2n. With a random pivot the O(n²) worst case is vanishingly unlikely. (`numpy.partition` / `argpartition` do this in C — it is how [top-k retrieval](01_big_o_and_timing.ipynb) avoided a full sort.)

In [ ]:
def kth_largest(nums, k, seed=0):
    a, pick = list(nums), random.Random(seed)
    target = len(a) - k                         # its index in ascending order
    lo, hi = 0, len(a) - 1
    while True:
        r = pick.randint(lo, hi)
        a[r], a[hi] = a[hi], a[r]
        p, i = a[hi], lo
        for j in range(lo, hi):                 # Lomuto partition of a[lo..hi]
            if a[j] < p:
                a[i], a[j] = a[j], a[i]
                i += 1
        a[i], a[hi] = a[hi], a[i]
        if i == target:
            return a[i]
        if i < target:
            lo = i + 1                          # the answer is to the right: drop the left side
        else:
            hi = i - 1


for (nums, k), expected in [(([3, 2, 1, 5, 6, 4], 2), 5), (([3, 2, 3, 1, 2, 4, 5, 5, 6], 4), 4), (([1], 1), 1), (([2, 2, 2], 2), 2)]:
    assert kth_largest(nums, k) == expected == heapq.nlargest(k, nums)[-1] == sorted(nums)[-k]
for _ in range(500):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 30))]
    k = rng.randint(1, len(nums))
    assert kth_largest(nums, k) == sorted(nums)[-k] == heapq.nlargest(k, nums)[-1]
print("kth_largest (quickselect) agrees with sorted()[-k] and heapq.nlargest on 500 random inputs")

## Timing: random vs already-sorted input

All eight sorts on random and on already-sorted lists of 100–800 numbers (small, because the O(n²) ones are slow in pure Python). Watch three things: the O(n²) sorts separate from the O(n log n) ones; on sorted input insertion and bubble sort become *linear* while naive quicksort becomes the *slowest*; and the C-implemented `sorted()` wins everywhere.

In [ ]:
def interleaved_times(calls, number=1, rounds=2):
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            t0 = time.perf_counter()
            for _ in range(number):
                call()
            best[i] = min(best[i], (time.perf_counter() - t0) / number)
    return best


algorithms = {"bubble": bubble_sort, "selection": selection_sort, "insertion": insertion_sort, "merge": merge_sort,
              "quick (last pivot)": lambda a: quick_sort(a, "last")[0], "quick (random pivot)": lambda a: quick_sort(a, "random")[0],
              "heap (heapq)": heap_sort, "sorted() (Timsort)": sorted}
sizes = [100, 200, 400, 800]
inputs = {"random": {n: random.Random(n).sample(range(n), n) for n in sizes}, "already sorted": {n: list(range(n)) for n in sizes}}
t_start = time.perf_counter()
timings = {kind: {name: interleaved_times([lambda f=f, a=data[n]: f(a) for n in sizes]) for name, f in algorithms.items()}
           for kind, data in inputs.items()}
print(f"timed {len(algorithms) * len(sizes) * 2} (algorithm, size, input) combinations in {time.perf_counter() - t_start:.1f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, kind in zip(axes, timings):
    for (name, ts), color in zip(timings[kind].items(), COLORS):
        ax.loglog(sizes, np.array(ts) * 1e3, marker="o", ms=4, color=color, label=name)
    ax.set(title=f"{kind} input", xlabel="n (log scale)")
    ax.minorticks_off()
    ax.set_xticks(sizes, [str(s) for s in sizes])
axes[0].set_ylabel("time, ms (log scale)")
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.suptitle("Eight sorts on random vs already-sorted input")
plt.tight_layout()
plt.show()

t_r, t_s = timings["random"], timings["already sorted"]
print(f"{'n = 800':>22} {'random':>10} {'sorted':>10}")
for name in algorithms:
    print(f"{name:>22} {t_r[name][-1] * 1e3:>8.2f}ms {t_s[name][-1] * 1e3:>8.2f}ms")
assert t_s["insertion"][-1] * 10 < t_r["insertion"][-1]                 # O(n) vs O(n²)
assert t_s["quick (last pivot)"][-1] > 5 * t_r["quick (last pivot)"][-1]  # the worst case, measured
assert all(t_r["sorted() (Timsort)"][-1] < t_r[name][-1] for name in algorithms if name != "sorted() (Timsort)")

Compare like with like: `heapq` and `sorted()` run in C, which is why the heap sort beats the pure-Python merge and quick sorts here even though it is usually the slowest O(n log n) sort when all are written in the same language. In an interview, write your own sort only when asked to; otherwise call `sorted()`/`.sort()` and say "O(n log n), stable, O(n) on nearly sorted data".

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| pairs / triplets / closest / intervals / greedy "earliest first" | sort first (O(n log n)), then scan or two pointers |
| sort by several fields | tuple key or `itemgetter(a, b)`; negate a numeric field to reverse it |
| reverse one non-numeric field | two stable passes (secondary first) |
| a pairwise ordering rule ("a before b if a+b > b+a") | `functools.cmp_to_key` |
| k-th largest / smallest, top-k | quickselect (O(n) avg), `heapq.nlargest` (O(n log k)), `np.argpartition` |
| integers in a small range | counting sort, O(n + k) |
| "how unsorted is it", ranking agreement | count inversions with merge sort (→ Kendall's τ) |
| nearly sorted input | insertion sort / Timsort are near-linear |
| data too big for memory | external merge sort: sort chunks, then `heapq.merge` them |
| guaranteed O(n log n) + O(1) space | heap sort |
| "is quicksort safe here?" | random pivot; three-way partition for many duplicates |

## Try it yourself

**1. Sort colours with counting sort.** Sort `[2,0,2,1,1,0]` using `counting_sort` from above (two passes, O(n)), and check it against the one-pass Dutch flag answer `[0,0,1,1,2,2]`.

In [ ]:
# Solution — try it yourself first, then run this
colors = [2, 0, 2, 1, 1, 0]
assert counting_sort(colors, k=3) == [0, 0, 1, 1, 2, 2]
for _ in range(200):
    case = [rng.randint(0, 2) for _ in range(rng.randint(0, 20))]
    assert counting_sort(case, k=3) == sorted(case)
print("counting_sort sorts 0/1/2 arrays in O(n + 3):", counting_sort(colors, k=3))

**2. H-index (LC 274).** A researcher has index h if h of their papers have at least h citations each. `[3,0,6,1,5]` → 3. Do it (a) by sorting descending, O(n log n), and (b) with counting buckets, O(n) — citations above n can be capped at n.

In [ ]:
# Solution — try it yourself first, then run this
def h_index_sort(citations):
    ranked = sorted(citations, reverse=True)
    return sum(1 for i, c in enumerate(ranked) if c >= i + 1)      # ranked[i] >= i + 1 holds for a prefix


def h_index_counting(citations):
    n = len(citations)
    buckets = [0] * (n + 1)
    for c in citations:
        buckets[min(c, n)] += 1                                    # cap at n: the answer is at most n
    papers = 0
    for h in range(n, -1, -1):                                     # papers with >= h citations
        papers += buckets[h]
        if papers >= h:
            return h
    return 0


for cites, expected in [([3, 0, 6, 1, 5], 3), ([1, 3, 1], 1), ([0], 0), ([100], 1), ([], 0), ([4, 4, 4, 4], 4)]:
    assert h_index_sort(cites) == h_index_counting(cites) == expected, cites
for _ in range(300):
    cites = [rng.randint(0, 12) for _ in range(rng.randint(0, 12))]
    assert h_index_sort(cites) == h_index_counting(cites)
print("h_index: the sorting and counting versions agree")

**3. Merge k sorted lists — the external-sort step.** Given several sorted lists (think: sorted chunks written to disk), produce one sorted stream. Use `heapq.merge`, check it against `sorted`, and say its complexity.

In [ ]:
# Solution — try it yourself first, then run this
chunks = [sorted(rng.randint(0, 1000) for _ in range(rng.randint(0, 50))) for _ in range(6)]
merged = list(heapq.merge(*chunks))        # lazy: holds one item per chunk in a heap -> O(N log k) time, O(k) memory
assert merged == sorted(x for chunk in chunks for x in chunk)
print(f"merged {len(chunks)} sorted chunks, {len(merged)} items; first ten: {merged[:10]}")

## Say it in an interview

- **30-second answer:** "Comparison sorts need Ω(n log n) comparisons. Merge sort guarantees O(n log n) and is stable but needs O(n) extra space; quicksort is in place and fastest on average, but a bad pivot makes it O(n²) — so I pick the pivot at random. Python uses Timsort: stable, O(n log n), and O(n) on already-sorted data because it merges existing runs."
- **Stability:** "equal keys keep their order — that is what lets two stable passes implement a two-key sort, and why `sorted(..., reverse=True)` still keeps ties in order."
- **Key numbers:** quicksort on sorted input with a last-element pivot does exactly n(n−1)/2 comparisons; a random pivot needs about 2n ln n ≈ 1.39 n log₂ n for large n; Timsort on sorted input needs n − 1; quickselect is ~2n expected.
- **Follow-ups:** "Sort 10 GB with 1 GB of RAM?" — external merge sort: sort 1 GB chunks, then k-way merge with a heap. "Integers 0–100?" — counting sort, O(n). "Top-k only?" — heap (O(n log k)) or quickselect (O(n)) instead of a full sort.
- **Traps:** calling selection/quick/heap sort stable; forgetting quicksort's O(log n) stack space (O(n) in the worst case); `list.sort()` returns `None`; negating a string key (impossible — use two passes or `cmp_to_key`); Lomuto partition degrading on many duplicates.

## Next

- [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) — merge sort and quicksort are recursion in action
- [10 · Heaps and top-k](10_heaps_and_top_k.ipynb) — heap sort, `nlargest`, k-way merge
- [13 · Greedy and intervals](13_greedy_and_intervals.ipynb) — "sort by end time" and friends
- GenAI links: [Reranking](../09_rag/05_reranking.ipynb) (sorting candidates by a better score) · [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) (ranking metrics)
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 10, "Sorting & searching") · [interview bank](../../ai_interview_prep/index.html)